In [683]:
!pip install -q -U langchain-google-genai newsapi-python streamlit pyngrok

In [684]:
!pip install -q -U langchain langchain-core

In [685]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [686]:
from google.colab import userdata

from newsapi import NewsApiClient

# 1. Retrieve the keys
gemini_key = userdata.get('GEMINI_API_KEY')
news_key = userdata.get('NEWS_API_KEY')

# 2. Check for missing or placeholder values
if not gemini_key or gemini_key == 'GEMINI_API_KEY':
    print("GEMINI_API_KEY is either missing or set to a placeholder name.")
else:
    print("GEMINI_API_KEY retrieved successfully.")

if not news_key or news_key == 'NEWS_API_KEY':
    print("NEWS_API_KEY is either missing or set to a placeholder name.")
else:
    print("NEWS_API_KEY retrieved successfully.")

# 3. Test Gemini API connectivity with a simple query
if gemini_key and gemini_key != 'GEMINI_API_KEY':
    try:
        # Using a verified model name like 'gemini-3.5-flash-lite'
        test_llm = ChatGoogleGenerativeAI(model='gemini-3.5-flash-lite', api_key=gemini_key)
        test_response = test_llm.invoke("Hello, is this key working?")
        print("Gemini API is working properly! Response received.")
    except Exception as e:
        print(f"Gemini API Key test failed: {e}")

# 4. Test NewsAPI connectivity
if news_key and news_key != 'NEWS_API_KEY':
    try:
        test_newsapi = NewsApiClient(api_key=news_key)
        # Test with a minor source fetch to verify connectivity
        test_newsapi.get_sources(language='en')
        print("NewsAPI is working properly!")
    except Exception as e:
        print(f"NewsAPI Key test failed: {e}")

GEMINI_API_KEY retrieved successfully.
NEWS_API_KEY retrieved successfully.
Gemini API is working properly! Response received.
NewsAPI is working properly!


In [687]:
def get_news_articles(query):
    articles = newsapi.get_everything(q=query,language="en",sort_by="relevancy",page_size=10)
    return articles["articles"]

In [688]:
from newsapi import NewsApiClient

# Re-initialize NewsApiClient using the validated 'news_key' from your verified tests
if 'news_key' in globals() and news_key:
    newsapi = NewsApiClient(api_key=news_key)

query = "Reliance Industries"
articles = get_news_articles(query)
print("Total articles fetched:", len(articles))
for i, article in enumerate(articles[:5], 1):
    print("\nArticle:", i)
    print("Title:", article["title"])
    print("Source:", article["source"]["name"])
    print("Published:", article["publishedAt"])
    print("Description:", article["description"])
    print("URL:", article["url"])

Total articles fetched: 10

Article: 1
Title: Elon Musk is beefing with telecom billionaire Mukesh Ambani as Starlink’s launch in India stalls. Here's why.
Source: Business Insider
Published: 2026-10-09T23:38:23Z
Description: Elon Musk called Mukesh Ambani, chairman of Reliance Industries, the "real boss of India," as he criticizes delays to launch Starlink in India.
URL: https://www.businessinsider.com/elon-musk-mukesh-ambani-beef-starlink-reliance-industries-jio-2026-10#article

Article: 2
Title: Top YouTuber MrBeast has named his first chief marketing officer
Source: Business Insider
Published: 2026-10-30T16:23:04Z
Description: YouTube star MrBeast's company named Michelle St. Jacques to the role, an expansion of her duties leading Feastables.
URL: https://www.businessinsider.com/mrbeast-jimmy-donaldson-first-chief-marketer-feastables-michelle-st-jacques-2026-10#article

Article: 3
Title: India's largest share sale could end cheap mobile internet for millions
Source: BBC News
Publis

In [689]:
def summarize_articles(articles):
    summaries = []
    for article in articles:
        title = article.get("title", "")
        description = article.get("description", "")
        content = article.get("content", "")
        text = f"""
        Title: {title}
        Description: {description}
        Content: {content}
        """
        summaries.append(text)
    return "\n\n".join(summaries)

In [690]:
summaries = summarize_articles(articles)
print(summaries[:5000])


        Title: Elon Musk is beefing with telecom billionaire Mukesh Ambani as Starlink’s launch in India stalls. Here's why.
        Description: Elon Musk called Mukesh Ambani, chairman of Reliance Industries, the "real boss of India," as he criticizes delays to launch Starlink in India.
        Content: Elon Musk asked on X if Mukesh Ambani, chairman of Reliance Industries, is the "real boss of India" as SpaceX's Starlink awaits regulatory approval in the country.Getty Images
<ul><li>Starlink has b… [+2803 chars]
        


        Title: Top YouTuber MrBeast has named his first chief marketing officer
        Description: YouTube star MrBeast's company named Michelle St. Jacques to the role, an expansion of her duties leading Feastables.
        Content: YouTuber Jimmy Donaldson, a.k.a. MrBeast, is expanding his company's C-suite, naming his first CMO.Sujit JAISWAL / AFP via Getty Images
<ul><li>The world's biggest creator, MrBeast, has found his fi… [+2717 chars]
        


      

In [691]:
template = """ You are an AI assistant helping an equity research analyst. Given the following query, summarize the most relevant news articles. Query: {query} """
prompt = ChatPromptTemplate.from_template(template)

print("Prompt template created successfully")


Prompt template created successfully


In [692]:
llm_chain = ( prompt | llm| StrOutputParser())
print("Gemini LangChain chain created successfully")

Gemini LangChain chain created successfully


In [693]:
def get_summary(query):
    # Fetch news
    articles = get_news_articles(query)
    if not articles:
        return "No relevant news articles found."
    # Extract article information
    summaries = summarize_articles(articles)
    # Generate AI analysis
    response = llm_chain.invoke({ "query": query, "summaries": summaries})
    return response

In [694]:
query = "Tata Motors"
final_summary = get_summary(query)
print(final_summary)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on recent financial and market news regarding **Tata Motors**, here is a summary of the most relevant developments categorized by key themes for your equity research:

### 1. Financial Performance & Earnings Trends
* **Demand Moderation in Domestic CVs:** Analysts have noted a cyclical slowdown in the Commercial Vehicle (CV) segment in India, impacting volume growth. However, fleet utilization rates remain stable, preventing a sharp drop in revenues.
* **JLR (Jaguar Land Rover) Profitability:** JLR continues to be the primary profit driver for Tata Motors, supported by a favorable product mix (higher sales of high-margin models like the Range Rover, Range Rover Sport, and Defender). Supply chain constraints have largely normalized, leading to better realization.
* **EBITDA Margins:** Management’s focus on deleveraging the balance sheet remains on track, with the standalone automotive business aiming for net-debt-free status, supported by steady margin expansion in both PV (Passen

In [695]:
!grep -n "gemini" /content/langchain_config.py

10:llm = ChatGoogleGenerativeAI(model = "gemini-3.8-flash-lite",temperature=0.2,api_key=GEMINI_API_KEY)
37:    gemini_key = os.getenv("GEMINI_API_KEY") or userdata.get("GEMINI_API_KEY")
40:    if not gemini_key or not news_key:
63:        model="gemini-2.5-flash",
64:        google_api_key=gemini_key,


In [696]:
from pathlib import Path

path = Path("/content/langchain_config.py")
code = path.read_text()

code = code.replace(
    "gemini-2.5-flash",
    "gemini-3.8-flash"
)

path.write_text(code)

print("Model configuration updated.")

Model configuration updated.


In [697]:
!grep -n "gemini" /content/langchain_config.py

10:llm = ChatGoogleGenerativeAI(model = "gemini-3.8-flash-lite",temperature=0.2,api_key=GEMINI_API_KEY)
37:    gemini_key = os.getenv("GEMINI_API_KEY") or userdata.get("GEMINI_API_KEY")
40:    if not gemini_key or not news_key:
63:        model="gemini-3.8-flash",
64:        google_api_key=gemini_key,


In [698]:
!pkill -f streamlit

In [699]:
%%writefile langchain_config.py
import os
from newsapi import NewsApiClient
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
NEWS_API_KEY = os.environ.get("NEWS_API_KEY")

llm = ChatGoogleGenerativeAI(model = "gemini-3.8-flash-lite",temperature=0.2,api_key=GEMINI_API_KEY)
newsapi = NewsApiClient(api_key=NEWS_API_KEY)
def get_news_articles(query):
    result=newsapi.get_everything(q=query,language="en",sort_by="relevancy",page_size=10)
    return result

def summarize_articles (articles):
    summaries = []
    for article in articles:
        text = f"""
        title = article.get("title","")
        Source: {article.get("source",{}.get ('name'))}
        Date : {article.get("publishedAt","")}
        Description: {article.get("description","")}
        URL: {article.get("url","")}
        Published At: {article.get("publishedAt","")}
        """
        summaries.append(text)
    return "\n\n".join(summaries)

Overwriting langchain_config.py


In [700]:
llm_chain = (prompt |llm | StrOutputParser())
def analyze_news(query):
    articles = get_news_articles(query)
    if not articles:
        return[], "No relevant news articles found."
    summaries = summarize_articles(articles)
    response = llm_chain.invoke ({"query":query,"summaries":summaries})
    return articles,response

In [701]:
!cat /content/langchain_config.py

import os
from newsapi import NewsApiClient
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
NEWS_API_KEY = os.environ.get("NEWS_API_KEY")

llm = ChatGoogleGenerativeAI(model = "gemini-3.8-flash-lite",temperature=0.2,api_key=GEMINI_API_KEY)
newsapi = NewsApiClient(api_key=NEWS_API_KEY)
def get_news_articles(query):
    result=newsapi.get_everything(q=query,language="en",sort_by="relevancy",page_size=10)
    return result

def summarize_articles (articles):
    summaries = []
    for article in articles:
        text = f"""
        title = article.get("title","")
        Source: {article.get("source",{}.get ('name'))}
        Date : {article.get("publishedAt","")}
        Description: {article.get("description","")}
        URL: {article.get("url","")}
        Published At: {article.get("publishedAt","")}
        """


In [702]:
import importlib
import langchain_config

importlib.reload(langchain_config)

print(langchain_config.analyze_news.__code__.co_firstlineno)

31


In [703]:
from pathlib import Path

path = Path("/content/langchain_config.py")
code = path.read_text()

if "def analyze_news(" not in code:
    with path.open("a") as f:
        f.write('''

def analyze_news(query):
    import os
    from google.colab import userdata
    from newsapi import NewsApiClient
    from langchain_google_genai import ChatGoogleGenerativeAI

    gemini_key = os.getenv("GEMINI_API_KEY") or userdata.get("GEMINI_API_KEY")
    news_key = os.getenv("NEWS_API_KEY") or userdata.get("NEWS_API_KEY")

    if not gemini_key or not news_key:
        raise ValueError("Gemini or NewsAPI key is missing")

    newsapi = NewsApiClient(api_key=news_key)

    result = newsapi.get_everything(
        q=query,
        language="en",
        sort_by="publishedAt",
        page_size=5
    )

    articles = result.get("articles", [])

    news_text = "\\\\n".join(
        f"Title: {a.get('title')}\\\\nDescription: {a.get('description')}"
        for a in articles
    )

    if not articles:
        return [], "No news articles found for this query."

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        google_api_key=gemini_key,
        temperature=0.2
    )

    prompt = f"""
    You are an equity research analyst.

    Analyze the following news about {query}.

    Provide:
    1. Executive summary
    2. Important news developments
    3. Potential impact on the company
    4. Positive and negative factors
    5. Key risks
    6. Overall outlook

    Base your analysis only on the supplied articles.
    Clearly distinguish reported facts from interpretations.
    Do not invent financial figures.

    News:
    {news_text}
    """

    response = llm.invoke(prompt)

    return articles, response.content
''')

    print("analyze_news function added successfully.")
else:
    print("analyze_news function already exists.")

analyze_news function added successfully.


In [704]:
%%writefile app.py
import streamlit as st
from langchain_config import analyze_news

st.set_page_config(page_title="Equity Research News Tool",page_icon="📊",layout="wide")
st.title("📊 Equity Research News Tool")
st.write( "AI-powered financial news research " "using Google Gemini and NewsAPI.")
query = st.text_input("Enter Company Name or News Topic", placeholder="Example: Reliance Industries")
if st.button("Get News", type="primary"):
    if query.strip():
        try:
            with st.spinner("Fetching news and analysing with Gemini..." ):articles, response = analyze_news(query)
            st.subheader("AI Research Summary")
            st.markdown(response)
            st.divider()
            st.subheader("Related News Articles")
            if articles:
                for article in articles:
                    st.markdown(f"### {article.get('title', 'Untitled')}")
                    st.write( "Source:",article.get("source", {}).get("name"))
                    st.write( "Published:",article.get("publishedAt") )
                    st.write( article.get("description"))
                    st.markdown( f"[Read Full Article]({article.get('url')})")
                    st.divider()
            else:
                st.info("No realted news article found.")
        except Exception as e:
            st.error(f"An error occurred: {e}")
    else:
        st.warning("Please enter a company name or news topic.")

Overwriting app.py


In [705]:
%%writefile requirements.txt
langchain
langchain-core
langchain-google-genai
newsapi-python
streamlit
pyngrok

Overwriting requirements.txt


In [706]:
from pyngrok import ngrok
from google.colab import userdata
import os

NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")

try:
    # Set auth token. If it is already configured, pyngrok might raise an error which we can ignore.
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("Ngrok configured successfully!")
except Exception as e:
    if "Authtoken already set" in str(e):
        print("Ngrok was already configured previously. Proceeding!")
    else:
        print(f"Handled expected config warning: {e}")

Ngrok was already configured previously. Proceeding!


In [707]:
import subprocess
import time
# Start Streamlit server
process = subprocess.Popen(["streamlit","run","app.py","--server.port","8501","--server.address","0.0.0.0"],stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)
time.sleep(5)
print("Streamlit server started")

Streamlit server started


In [708]:
from pyngrok import ngrok

try:
    # Disconnect any existing open tunnels to free up the limit
    tunnels = ngrok.get_tunnels()
    for tunnel in tunnels:
        print(f"Closing active tunnel: {tunnel.public_url}")
        ngrok.disconnect(tunnel.public_url)

    # Kill current background ngrok processes just in case
    ngrok.kill()
except Exception as e:
    print(f"No previous tunnels to clear: {e}")

# Open a fresh tunnel
try:
    public_url = ngrok.connect(8501)
    print("\nYour application is running at:")
    print(public_url)
except Exception as e:
    print(f"\nFailed to start tunnel: {e}")

Closing active tunnel: https://blooper-enlisted-flashing.ngrok-free.dev

Your application is running at:
NgrokTunnel: "https://blooper-enlisted-flashing.ngrok-free.dev" -> "http://localhost:8501"


In [709]:
%%writefile README.md

# Equity Research News Tool

## Project Overview

This project develops an AI-powered financial
news research application.

## Technologies Used

- Python
- Google Colab
- Google Gemini API
- LangChain
- NewsAPI
- Streamlit

## Features

1. Search financial news by company name.
2. Fetch relevant news articles.
3. Summarize articles using Gemini.
4. Identify positive and negative market signals.
5. Generate investor-oriented insights.
6. Display original news sources.

## Execution

1. Install requirements.
2. Configure API keys.
3. Run langchain_config.py through app.py.
4. Launch Streamlit.
5. Search for a company or financial topic.

Overwriting README.md


In [710]:
!nl -ba /content/langchain_config.py | sed -n '1,90p'

     1	import os
     2	from newsapi import NewsApiClient
     3	from langchain_google_genai import ChatGoogleGenerativeAI
     4	from langchain_core.prompts import ChatPromptTemplate
     5	from langchain_core.output_parsers import StrOutputParser
     6	
     7	GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
     8	NEWS_API_KEY = os.environ.get("NEWS_API_KEY")
     9	
    10	llm = ChatGoogleGenerativeAI(model = "gemini-3.8-flash-lite",temperature=0.2,api_key=GEMINI_API_KEY)
    11	newsapi = NewsApiClient(api_key=NEWS_API_KEY)
    12	def get_news_articles(query):
    13	    result=newsapi.get_everything(q=query,language="en",sort_by="relevancy",page_size=10)
    14	    return result
    15	
    16	def summarize_articles (articles):
    17	    summaries = []
    18	    for article in articles:
    19	        text = f"""
    20	        title = article.get("title","")
    21	        Source: {article.get("source",{}.get ('name'))}
    22	        Date : {article.get("publishedAt","")}
  